In [1]:
#| output: true
#| warning: false
#| eval: true
#| echo: false

import pandas as pd
import numpy as np
import plotly.express as px

filename = "../private/CARD Group Timeline.xlsx"

df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

names = df['Display Name'].values

# Get the current group members and alumni
current_group_member_indices = df[df.apply(
    lambda row: row.astype(str).str.contains('current').any(),
      axis=1)].index.tolist()
alumni_indices = df[~df.index.isin(current_group_member_indices)].index.tolist()

current_group_members = df.loc[current_group_member_indices, 'Display Name']
alumni = df.loc[alumni_indices, 'Display Name']

current_index = np.zeros([len(names), 1])
current_index[current_group_member_indices] = 1
alumni_index = np.zeros([len(names), 1])
alumni_index[alumni_indices] = 1
df['current'] = current_index
df['alumni'] = alumni_index
del current_index, alumni_index

# Convert Finish and Start columns to datetime
for col in np.ravel([[col for col in df.columns if 'Finish' in col],
              [col for col in df.columns if 'Start' in col]]):
    df[col] = pd.to_datetime(df[col], errors='coerce', format='%Y-%m-%d %H:%M:%S')

# Replace 'current' with today's date in the DataFrame
# This assumes 'current' is used in the 'Ultimate Role Finish' column
df = df.mask(df == 'current', pd.to_datetime(pd.Timestamp.today()))

# Fill in remaining dates with current date time
for col in np.ravel([[col for col in df.columns if 'Finish' in col],
              [col for col in df.columns if 'Start' in col]]):
    df.loc[df[col].isna(),col] = \
        pd.to_datetime(pd.Timestamp.today())


#| output: false

rolehistory = ['Ultimate', 'Penultimate', 'Antepenultimate', 'Preantepenultimate', 'Propreantepenultimate', 'Ultrasuprapropreantepenultimate']

role_aliases = {
    'BS - Undergraduate Full-Time Research (Paid)': 'BS - Research Experience for Undergraduates (Full-Time Paid)',
    'BS - Undergraduate Lab Assistant (Part-Time Paid)': 'BS - Undergraduate Part-Time Research (Paid)'
}

role_styles = {
    'Postdoctoral Researcher': {'color': 'blueviolet', 'pattern': '', 'rank': 1000},
    'PhD - Primary Advisee': {'color': 'firebrick', 'pattern': '', 'rank': 750},
    'PhD - Secondary Advisee': {'color': 'firebrick', 'pattern': '/', 'rank': 350},
    'PhD - Committee Member': {'color': 'firebrick', 'pattern': 'x', 'rank': 225},
    'MS - Primary Advisee': {'color': 'mediumblue', 'pattern': '', 'rank': 500},
    'MS - Secondary Advisee': {'color': 'mediumblue', 'pattern': '/', 'rank': 175},
    'MS - Committee Member': {'color': 'mediumblue', 'pattern': 'x', 'rank': 75},
    'MEng - Primary Advisee': {'color': 'lightseagreen', 'pattern': '', 'rank': 225},
    'MEng - Secondary Advisee': {'color': 'lightseagreen', 'pattern': '/', 'rank': 2},
    'BS - Undergrad Co-Op Research Fellowship': {'color': 'forestgreen', 'pattern': '', 'rank': 200},
    'BS - Undergraduate Part-Time Research (Paid)': {'color': 'forestgreen', 'pattern': '/', 'rank': 20},
    'BS - Undergraduate Independent Study Researcher (Part Time for Credit)': {'color': 'forestgreen', 'pattern': '|', 'rank': 10},
    'BS - Senior Capstone Project': {'color': 'forestgreen', 'pattern': '+', 'rank': 30},
    'BS - Experiential Exploration Program Researcher (Full-Time for Credit)': {'color': 'forestgreen', 'pattern': '.', 'rank': 75},
    'BS - Research Experience for Undergraduates (Full-Time Paid)': {'color': 'forestgreen', 'pattern': '-', 'rank': 150},
    'BS - Research Volunteer': {'color': 'forestgreen', 'pattern': 'x', 'rank': 1},
}

def canonicalize_role(degree_role):
    if pd.isna(degree_role):
        return 'Unknown'
    role = str(degree_role).strip()
    return role_aliases.get(role, role)

def get_role_style(degree_role):
    canonical = canonicalize_role(degree_role)
    return canonical, role_styles.get(canonical, {'color': 'magenta', 'pattern': 'x', 'rank': 1})

time_with_group = pd.to_timedelta(pd.Series(np.zeros(np.shape(names))), unit='s')
for role in rolehistory:
    time_with_group += pd.to_datetime(
        df['{0} Role Finish'.format(role)]) \
          - pd.to_datetime(df['{0} Role Start'.format(role)]
                           )
time_with_group = time_with_group / (np.timedelta64(1, 'D'))
df['time_with_group'] = time_with_group
rankvalue = []
for n in np.arange(0, len(names)):
    groupmember = df.loc[n]
    for role in rolehistory:
        canonical_role, style = get_role_style(groupmember['{0} Degree-Role'.format(role)])
        if role == 'Ultimate':
            rankvalue.append(style['rank'])

current = 4999 * df['current'].values + 1

value = current + rankvalue + 0.5 * time_with_group

# Sort in the order we want to plot
df['rankvalue'] = rankvalue

df['value'] = value
order = np.flipud(np.argsort(np.array(value)))
df['order'] = order

dfn = df.sort_values(by='value', ascending=False)
dfn.reset_index(inplace=True, drop=True)

names_sorted = dfn['Display Name'].values

timeline_records = []
for _, groupmember in dfn.iterrows():
    name = groupmember['Display Name']
    for role in rolehistory:
        degree_role = groupmember['{0} Degree-Role'.format(role)]
        canonical_role, style = get_role_style(degree_role)

        timeline_records.append({
            'name': name,
            'degree_role': canonical_role,
            'color': style['color'],
            'pattern': style['pattern'],
            'start': pd.to_datetime(groupmember['{0} Role Start'.format(role)]),
            'finish': pd.to_datetime(groupmember['{0} Role Finish'.format(role)])
        })

timeline_df = pd.DataFrame(timeline_records)

color_discrete_map = {k: v['color'] for k, v in role_styles.items()}
color_discrete_map['Unknown'] = 'magenta'

pattern_shape_map = {k: (v['pattern'] if v['pattern'] else '') for k, v in role_styles.items()}
pattern_shape_map['Unknown'] = 'x'

timeline_df['degree_role_clean'] = timeline_df['degree_role'].fillna('Unknown')

fig = px.timeline(
    timeline_df,
    x_start='start',
    x_end='finish',
    y='name',
    color='degree_role_clean',
    pattern_shape='degree_role_clean',
    category_orders={'name': list(names_sorted)},
    color_discrete_map=color_discrete_map,
    pattern_shape_map=pattern_shape_map
)

# Plotly 3.x can hide overlaid axes without a bound trace.
# Add a transparent helper trace on x2 so top ticks always render.
x_min = timeline_df['start'].min()
x_max = timeline_df['finish'].max()
y_ref = names_sorted[0] if len(names_sorted) else ''
fig.add_scatter(
    x=[x_min, x_max],
    y=[y_ref, y_ref],
    mode='lines',
    line=dict(color='rgba(0,0,0,0)', width=0),
    hoverinfo='skip',
    showlegend=False,
    xaxis='x2',
    yaxis='y'
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.6,
    hovertemplate='<b>%{y}</b><br>Role: %{fullData.name}<br>Start: %{base|%Y-%m-%d}<br>Finish: %{x|%Y-%m-%d}<extra></extra>'
)

fig.update_yaxes(
    autorange=True,
    title=None,
    showgrid=False,
    tickfont=dict(size=11),
    ticks='',
    automargin=True
)
fig.update_xaxes(
    title=None,
    side='bottom',
    showticklabels=True,
    ticks='outside',
    ticklabelposition='outside bottom',
    showgrid=True,
    griddash='dot',
    gridcolor='rgba(120,120,120,0.35)',
    tickfont=dict(size=11),
    tickformat='%b %Y',
    automargin=True
)
fig.update_layout(
    title=dict(text='CARD Group Timeline', x=0.5, xanchor='center', font=dict(size=16)),
    legend_title_text='',
    legend=dict(
        x=0.0,
        y=0.01,
        xanchor='left',
        yanchor='bottom',
        orientation='v',
        font=dict(size=11),
        bgcolor='rgba(255,255,255,0.7)'
    ),
    xaxis2=dict(
        matches='x',
        overlaying='x',
        side='top',
        visible=True,
        showticklabels=True,
        ticks='outside',
        ticklabelposition='outside top',
        layer='above traces',
        showgrid=False,
        automargin=True,
        tickfont=dict(size=11),
        tickformat='%b %Y',
        title=None
    ),
    template='simple_white',
    height=max(450, int(24 * len(names_sorted))),
    margin=dict(l=10, r=10, t=100, b=100),
    bargap=0.08,
    hoverlabel=dict(font_size=11)
)

fig.show()

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed

